# Preparacion del conjunto de datos (Silver -> Gold con Spark)

Objetivo: preparar el dataset etiquetado de mood para entrenamiento.

Alcance: limpieza, consistencia, tratamiento de outliers, features derivadas, escalado y split.

Fuente unica: capa Silver en S3. Procesamiento completo con Apache Spark.

Salida: dataset preparado en S3 (Gold).

In [2]:
import os
import sys
import shutil
from pathlib import Path
from typing import Dict, List, Tuple

import boto3
import numpy as np
import pandas as pd
from pyspark.sql import functions as F

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))
from src.config import load_settings
from src.spark.session import build_spark

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 120)

settings = load_settings()
spark = build_spark("music-recommender-data-prep")

# Temporary local folder for Silver data downloaded from S3
local_silver_base = settings.project_root / "data_lake" / "tmp_silver"
local_silver_mood = local_silver_base / "mood_dataset"
local_silver_tracks = local_silver_base / "tracks_dataset"
local_silver_mood.mkdir(parents=True, exist_ok=True)
local_silver_tracks.mkdir(parents=True, exist_ok=True)

s3 = boto3.client("s3", region_name=settings.aws_region)

def download_prefix(bucket: str, prefix: str, local_dir: Path) -> None:
    if local_dir.exists():
        shutil.rmtree(local_dir)
    local_dir.mkdir(parents=True, exist_ok=True)
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            if key.endswith("/"):
                continue
            rel = key[len(prefix):]
            if not rel:
                continue
            target = local_dir / rel
            target.parent.mkdir(parents=True, exist_ok=True)
            s3.download_file(bucket, key, str(target))

settings

Settings(project_root=WindowsPath('C:/Users/losal/Desktop/CursoEspecializacion/TFG/Music_Mood_Activity_Recommender'), data_dir=WindowsPath('C:/Users/losal/Desktop/CursoEspecializacion/TFG/Music_Mood_Activity_Recommender/datasets'), local_lake_dir=WindowsPath('C:/Users/losal/Desktop/CursoEspecializacion/TFG/Music_Mood_Activity_Recommender/data_lake'), bucket_name='music-recommender-data-lake-246681677043', aws_region='us-east-1', s3_endpoint_url=None, use_s3=True, kafka_bootstrap_servers='localhost:9092', kafka_group_id='music-recommender-pipeline', max_rows_per_dataset=9999999999999999999999999999999999999999999, producer_batch_size=5000, producer_batch_delay_seconds=0.1, consumer_timeout_seconds=20, mongo_uri='mongodb://localhost:27017', mongo_database='music_recommender', rds_dsn='dbname=music_recommender user=postgres password=postgres host=localhost port=5432')

In [3]:
silver_prefix_mood = "silver/mood_dataset/"
silver_prefix_tracks = "silver/tracks_dataset/"
download_prefix(settings.bucket_name, silver_prefix_mood, local_silver_mood)
download_prefix(settings.bucket_name, silver_prefix_tracks, local_silver_tracks)

mood_silver = spark.read.parquet(str(local_silver_mood))
tracks_silver = spark.read.parquet(str(local_silver_tracks))

print("Silver mood:", f"s3://{settings.bucket_name}/{silver_prefix_mood}")
print("Silver tracks:", f"s3://{settings.bucket_name}/{silver_prefix_tracks}")
print("Local temp (mood):", local_silver_mood)
print("Local temp (tracks):", local_silver_tracks)
print("Rows mood:", mood_silver.count())
print("Rows tracks:", tracks_silver.count())

Silver mood: s3://music-recommender-data-lake-246681677043/silver/mood_dataset/
Silver tracks: s3://music-recommender-data-lake-246681677043/silver/tracks_dataset/
Local temp (mood): C:\Users\losal\Desktop\CursoEspecializacion\TFG\Music_Mood_Activity_Recommender\data_lake\tmp_silver\mood_dataset
Local temp (tracks): C:\Users\losal\Desktop\CursoEspecializacion\TFG\Music_Mood_Activity_Recommender\data_lake\tmp_silver\tracks_dataset
Rows mood: 160000
Rows tracks: 54160


## 1. Comprension general (mood)
Se valida estructura, tipos y nulos del dataset etiquetado.

In [4]:
def overview_spark(df, name: str, preview_rows: int = 3):
    print(name, "rows", df.count())
    display(df.limit(preview_rows).toPandas())
    df.printSchema()
    null_exprs = [F.mean(F.col(c).isNull().cast("double")).alias(c) for c in df.columns]
    nulls = df.select(*null_exprs).toPandas().T.rename(columns={0: "null_pct"})
    display(nulls.sort_values("null_pct", ascending=False).head(10))

overview_spark(mood_silver, "mood_silver")
overview_spark(tracks_silver, "tracks_silver")

mood_silver rows 160000


,acousticness,danceability,duration_ms,energy,instrumentalness,mood_label,liveness,loudness,spec_rate,speechiness,tempo,uri,valence
0,0.026700,0.402,89564,0.902,0.004120,2,0.1470,-3.105,2.847126e-06,0.2550,145.898,spotify:track:1lySGuhlcZUJxLA5W5Bo7k,0.5380
1,0.000109,0.498,209799,0.920,0.000002,2,0.1250,-4.183,3.417557e-07,0.0717,146.043,spotify:track:23tOVsU7dRtnTxupe6eQSd,0.6700
2,0.761000,0.115,212222,0.177,0.953000,3,0.0855,-17.873,1.837698e-07,0.0390,79.191,spotify:track:3SO3Me0mzDwvIkZ27TYXJr,0.0375


root
 |-- acousticness: double (nullable = true)
 |-- danceability: double (nullable = true)
 |-- duration_ms: long (nullable = true)
 |-- energy: double (nullable = true)
 |-- instrumentalness: double (nullable = true)
 |-- mood_label: integer (nullable = true)
 |-- liveness: double (nullable = true)
 |-- loudness: double (nullable = true)
 |-- spec_rate: double (nullable = true)
 |-- speechiness: double (nullable = true)
 |-- tempo: double (nullable = true)
 |-- uri: string (nullable = true)
 |-- valence: double (nullable = true)



,null_pct
acousticness,0.0
danceability,0.0
duration_ms,0.0
energy,0.0
instrumentalness,0.0
mood_label,0.0
liveness,0.0
loudness,0.0
spec_rate,0.0
speechiness,0.0


tracks_silver rows 54160


,acousticness,album_name,artists,danceability,duration_ms,energy,explicit,instrumentalness,key,liveness,loudness,mode,popularity,speechiness,tempo,time_signature,track_genre,track_id,track_name,valence
0,0.075700,Lolly,Rill,0.910,160725,0.374,True,0.00301,8,0.154,-9.844,0,44,0.1990,104.042,4,german,0000vdREvCVMxbQTkS888c,Lolly,0.432
1,0.004350,Discography 1991-1993,Dropdead,0.268,56200,0.968,False,0.77900,4,0.370,-11.194,1,14,0.1020,107.060,4,grindcore,005U5ZHLoA4OEkWewAa32S,Strength In Your Conviction,0.402
2,0.000897,Steelfactory,U.D.O.,0.420,242066,0.950,False,0.00374,6,0.336,-6.017,0,20,0.0418,85.991,4,heavy-metal,00C93bsNIjHStKvr1lPJee,Keeper of My Soul,0.501


root
 |-- acousticness: double (nullable = true)
 |-- album_name: string (nullable = true)
 |-- artists: string (nullable = true)
 |-- danceability: double (nullable = true)
 |-- duration_ms: integer (nullable = true)
 |-- energy: double (nullable = true)
 |-- explicit: boolean (nullable = true)
 |-- instrumentalness: double (nullable = true)
 |-- key: integer (nullable = true)
 |-- liveness: double (nullable = true)
 |-- loudness: double (nullable = true)
 |-- mode: integer (nullable = true)
 |-- popularity: integer (nullable = true)
 |-- speechiness: double (nullable = true)
 |-- tempo: double (nullable = true)
 |-- time_signature: integer (nullable = true)
 |-- track_genre: string (nullable = true)
 |-- track_id: string (nullable = true)
 |-- track_name: string (nullable = true)
 |-- valence: double (nullable = true)



,null_pct
acousticness,0.0
album_name,0.0
artists,0.0
danceability,0.0
duration_ms,0.0
energy,0.0
explicit,0.0
instrumentalness,0.0
key,0.0
liveness,0.0


## 2. Seleccion de variables base
Mood: se eliminan columnas no utiles (como `uri`).
Tracks: se conserva `track_id` y solo las columnas compartidas con mood para asegurar compatibilidad.

In [5]:
MOOD_FEATURES = [
    "danceability", "energy", "speechiness", "acousticness",
    "instrumentalness", "liveness", "valence", "loudness", "tempo", "spec_rate", "duration_ms"
 ]
TRACKS_FEATURES = [
    "danceability", "energy", "speechiness", "acousticness",
    "instrumentalness", "liveness", "valence", "loudness", "tempo", "duration_ms"
 ]

mood_features = [c for c in MOOD_FEATURES if c in mood_silver.columns]
tracks_features = [c for c in TRACKS_FEATURES if c in tracks_silver.columns]
shared_features = [c for c in mood_features if c in tracks_features]
if "mood_label" not in mood_silver.columns:
    raise ValueError("mood_label no existe en Silver")
if "track_id" not in tracks_silver.columns:
    raise ValueError("track_id no existe en Silver")

mood_base = mood_silver.select(["mood_label", *shared_features])
tracks_base = tracks_silver.select(["track_id", *shared_features])

# Normalizar tipos numericos y eliminar columnas no necesarias (uri)
for col in shared_features:
    mood_base = mood_base.withColumn(col, F.col(col).cast("double"))
    tracks_base = tracks_base.withColumn(col, F.col(col).cast("double"))
mood_base = mood_base.withColumn("mood_label", F.col("mood_label").cast("int"))

tracks_base

DataFrame[track_id: string, danceability: double, energy: double, speechiness: double, acousticness: double, instrumentalness: double, liveness: double, valence: double, loudness: double, tempo: double, duration_ms: double]

## 3. Limpieza: nulos e inconsistencias
Se corrigen tipos y se imputan nulos con medianas (sin eliminar mas filas de lo necesario).

In [6]:
def impute_with_median(df, numeric_cols: List[str]):
    medians = {}
    for col in numeric_cols:
        quantiles = df.approxQuantile(col, [0.5], 0.001)
        medians[col] = quantiles[0] if quantiles else None
    for col, median in medians.items():
        if median is not None:
            df = df.withColumn(col, F.coalesce(F.col(col), F.lit(median)))
    return df

# Mood: eliminar filas sin target
mood_clean = mood_base.filter(F.col("mood_label").isNotNull())
mood_clean = impute_with_median(mood_clean, shared_features)

# Tracks: eliminar filas sin track_id y duplicados por track_id
tracks_clean = tracks_base.filter(F.col("track_id").isNotNull()).dropDuplicates(["track_id"])
tracks_clean = impute_with_median(tracks_clean, shared_features)

print("Rows mood after cleaning:", mood_clean.count())
print("Rows tracks after cleaning:", tracks_clean.count())

Rows mood after cleaning: 160000
Rows tracks after cleaning: 54160


## 4. Outliers
Se aplica clipping IQR por feature para reducir extremos sin borrar registros.

In [7]:
def clip_outliers_iqr(df, cols: List[str]):
    for col in cols:
        q1 = df.approxQuantile(col, [0.25], 0.001)[0]
        q3 = df.approxQuantile(col, [0.75], 0.001)[0]
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        df = df.withColumn(col, F.when(F.col(col) < lower, lower).when(F.col(col) > upper, upper).otherwise(F.col(col)))
    return df

mood_clipped = clip_outliers_iqr(mood_clean, shared_features)
tracks_clipped = clip_outliers_iqr(tracks_clean, shared_features)

mood_clipped

DataFrame[mood_label: int, danceability: double, energy: double, speechiness: double, acousticness: double, instrumentalness: double, liveness: double, valence: double, loudness: double, tempo: double, duration_ms: double]

## 5. Ingenieria de features
Se mantiene el conjunto de columnas original para asegurar compatibilidad con el dataset de clasificacion.

In [8]:
# No se crean columnas nuevas para mantener compatibilidad con el dataset de clasificacion
mood_fe = mood_clipped
tracks_fe = tracks_clipped

mood_fe

DataFrame[mood_label: int, danceability: double, energy: double, speechiness: double, acousticness: double, instrumentalness: double, liveness: double, valence: double, loudness: double, tempo: double, duration_ms: double]

## 6. Escalado
Se estandarizan variables numericas con z-score usando estadisticas del dataset.

In [9]:
feature_cols = [c for c in mood_fe.columns if c != "mood_label"]

# Compute mean/std for scaling (from mood)
stats_rows = []
stats_map = {}
for col in feature_cols:
    row = mood_fe.agg(
        F.mean(F.col(col)).alias("mean"),
        F.stddev(F.col(col)).alias("std"),
    ).collect()[0]
    mean_val = row["mean"]
    std_val = row["std"] or 1.0
    stats_rows.append({"feature": col, "mean": float(mean_val), "std": float(std_val)})
    stats_map[col] = (mean_val, std_val)
    mood_fe = mood_fe.withColumn(col, (F.col(col) - F.lit(mean_val)) / F.lit(std_val))

mood_scaled = mood_fe
scaler_stats = pd.DataFrame(stats_rows)
scaler_stats.head()


,feature,mean,std
0,danceability,0.549246,0.190208
1,energy,0.561777,0.282446
2,speechiness,0.068179,0.044252
3,acousticness,0.392082,0.367320
4,instrumentalness,0.265951,0.377149


In [10]:
# Apply the same scaling to tracks (only shared features)
tracks_scaled = tracks_fe
for col, (mean_val, std_val) in stats_map.items():
    if col in tracks_scaled.columns:
        tracks_scaled = tracks_scaled.withColumn(col, (F.col(col) - F.lit(mean_val)) / F.lit(std_val))
tracks_scaled

DataFrame[track_id: string, danceability: double, energy: double, speechiness: double, acousticness: double, instrumentalness: double, liveness: double, valence: double, loudness: double, tempo: double, duration_ms: double]

## 7. Split train/test
Split estratificado 80/20 por `mood_label`.

In [11]:
labels = [row[0] for row in mood_scaled.select("mood_label").distinct().collect()]
train_parts = []
test_parts = []
for label in labels:
    subset = mood_scaled.filter(F.col("mood_label") == label)
    train_part, test_part = subset.randomSplit([0.8, 0.2], seed=42)
    train_parts.append(train_part)
    test_parts.append(test_part)

train_df = train_parts[0]
for part in train_parts[1:]:
    train_df = train_df.unionByName(part)
test_df = test_parts[0]
for part in test_parts[1:]:
    test_df = test_df.unionByName(part)

print("Train rows:", train_df.count(), "Test rows:", test_df.count())
train_df.groupBy("mood_label").count().show()
test_df.groupBy("mood_label").count().show()

Train rows: 128764 Test rows: 31236
+----------+-----+
|mood_label|count|
+----------+-----+
|         1|48273|
|         3|19623|
|         2|22607|
|         0|38261|
+----------+-----+

+----------+-----+
|mood_label|count|
+----------+-----+
|         1|11797|
|         3| 4743|
|         2| 5429|
|         0| 9267|
+----------+-----+



## 8. Guardado en Gold (S3)
Se escriben train/test y estadisticas de escalado en S3 Gold.

In [12]:
gold_local = settings.project_root / "data_lake" / "tmp_gold" / "mood_prepared"
tracks_local = settings.project_root / "data_lake" / "tmp_gold" / "tracks_prepared"
gold_tracks_local = settings.project_root / "data_lake" / "gold" / "tracks_dataset"
for path in [gold_local, tracks_local, gold_tracks_local]:
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

train_local = gold_local / "train"
test_local = gold_local / "test"
scaler_local = gold_local / "scaler_stats"
tracks_out_local = tracks_local / "full"

train_df.write.mode("overwrite").parquet(str(train_local))
test_df.write.mode("overwrite").parquet(str(test_local))

# Guardar scaler_stats sin usar Spark (evita fallos del worker).
scaler_local.mkdir(parents=True, exist_ok=True)
scaler_file = scaler_local / "scaler_stats.parquet"
try:
    scaler_stats.fillna(0).to_parquet(scaler_file, index=False)
except Exception:
    scaler_file = scaler_local / "scaler_stats.csv"
    scaler_stats.fillna(0).to_csv(scaler_file, index=False)

tracks_scaled.write.mode("overwrite").parquet(str(tracks_out_local))

# Guardar metadata de tracks (para el recomendador).
metadata_cols = [
    col
    for col in [
        "track_id",
        "track_name",
        "artists",
        "album_name",
        "track_genre",
        "popularity",
        "explicit",
    ]
    if col in tracks_silver.columns
 ]
tracks_silver.select(metadata_cols).write.mode("overwrite").parquet(str(gold_tracks_local))

def upload_directory(local_dir: Path, bucket: str, prefix: str) -> None:
    for path in local_dir.rglob("*"):
        if path.is_file():
            rel = path.relative_to(local_dir).as_posix()
            key = f"{prefix}/{rel}"
            s3.upload_file(str(path), bucket, key)

upload_directory(gold_local, settings.bucket_name, "gold/mood_prepared")
upload_directory(tracks_local, settings.bucket_name, "gold/tracks_prepared")
upload_directory(gold_tracks_local, settings.bucket_name, "gold/tracks_dataset")

print("Saved and uploaded to:")
print("-", f"s3://{settings.bucket_name}/gold/mood_prepared/train")
print("-", f"s3://{settings.bucket_name}/gold/mood_prepared/test")
print("-", f"s3://{settings.bucket_name}/gold/mood_prepared/scaler_stats")
print("-", f"s3://{settings.bucket_name}/gold/tracks_prepared/full")
print("-", f"s3://{settings.bucket_name}/gold/tracks_dataset")

Saved and uploaded to:
- s3://music-recommender-data-lake-246681677043/gold/mood_prepared/train
- s3://music-recommender-data-lake-246681677043/gold/mood_prepared/test
- s3://music-recommender-data-lake-246681677043/gold/mood_prepared/scaler_stats
- s3://music-recommender-data-lake-246681677043/gold/tracks_prepared/full
- s3://music-recommender-data-lake-246681677043/gold/tracks_dataset
